# 01 · Run Evaluation (Option 1: Deterministic + Optional Judge)

Execute an evaluation suite locally or in a Microsoft Fabric notebook. When live tenant credentials (`CFEVAL_TENANT_ID`, `CFEVAL_CLIENT_ID`, `CFEVAL_TEST_USER_UPN`) are absent, `MOCK` defaults automatically to `True` so every cell runs safely offline.

In [ ]:
import pathlib
import sys

repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

from cfeval.cli import build_harness
from cfeval.config import Settings

settings = Settings.load(env_file=str(repo / ".env"), output_dir=str(repo / "build" / "notebook-runs"))
has_live_creds = bool(settings.tenant_id and settings.client_id and settings.test_user_upn)
MOCK = True if not has_live_creds else True  # Set MOCK = False explicitly only when running against a live tenant
print(f"Loaded settings | has_live_creds={has_live_creds} | MOCK={MOCK}")

In [ ]:
outcome = build_harness(
    settings,
    option="fabric-native",
    mock=MOCK,
    publish=False,
    profile="tuned",
    repeats=1,
    run_id="nb01-eval-run",
    run_name="Notebook 01 Evaluation",
).run()

print(f"Run {outcome.run_id} completed with status={outcome.status}")
print("Aggregate metrics:", outcome.metrics)

In [ ]:
for row in outcome.results:
    print(
        f"{row['QuestionId']:4s} | {row['Verdict']:12s} | "
        f"val={row['ValueScore']} art={row['ArtifactScore']} mod={row['ModelScore']} | "
        f"queried={row.get('QueriedModels') or '-'}"
    )